# M8 · Establish component collaboration

Integration runs real collaborating components; a substitute controls an external outcome. The temporary API/database and the mocked HTTP client establish different claims.

**Start:** Before class, from `current-course/teaching-snippets/`, run `uv sync --extra dev` and select its Python environment as the notebook kernel. Notebook-editor/kernel setup is separate classroom preparation. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

**Temporary resources:** The setup creates a fresh disposable SQLite file. The final storage step removes that experiment. If you stop early, rerun the cleanup cell when finished. Restart and run all for a new baseline; do not repeatedly insert the same stored ID or recreate the same tables within one run.

## Exploration 1 — build and observe the mechanism

### 1. Own a temporary database

Integration evidence needs real local resources with isolated lifetime. A connection is opened in the operation’s thread.

**Before running:** Predict the result or state change.

In [ ]:
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

directory = TemporaryDirectory()
database_path = Path(directory.name) / "shop.sqlite"
def connect():
    connection = sqlite3.connect(database_path)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    return closing(connection)
print(database_path.name)


**Change and explain:** Explain why this path must not be the regular Nano-Zon database.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The notebook owns a fresh temporary database path.

</details>

### 2. Supply the local storage structure

The schema is plumbing for this lesson. Real related writes let the later check establish collaboration and rollback.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    connection.executescript("""
        CREATE TABLE orders (order_id TEXT PRIMARY KEY, total_paise INTEGER);
        CREATE TABLE order_lines (
            order_id TEXT REFERENCES orders(order_id), line_number INTEGER,
            product_id TEXT, name TEXT, unit_price_paise INTEGER, quantity INTEGER,
            PRIMARY KEY (order_id, line_number)
        );
    """)
    print(connection.execute("SELECT COUNT(*) FROM orders").fetchone()[0])


**Change and explain:** Which later observation could establish that a route really persisted lines?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The temporary order store begins empty.

</details>

### 3. Supply a checked request boundary

Run validation for real when it is part of the integration claim. Controlled storage does not require replacing the contract.

**Before running:** Predict the result or state change.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class RequestedItem(BaseModel):
    product_id: str
    quantity: int = Field(strict=True, gt=0)

class OrderRequest(BaseModel):
    items: list[RequestedItem] = Field(min_length=1)

payload = {"items": [{"product_id": "1", "quantity": 2}, {"product_id": "3", "quantity": 1}]}
print(OrderRequest.model_validate(payload).model_dump())


**Change and explain:** Which request would be rejected before storage is called?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

A two-cable, one-bottle request satisfies the input constraints.

</details>

### 4. Keep the real domain result

Integration should use the operation whose output the route promises. The catalog supplies authoritative price snapshots.

**Before running:** Predict the result or state change.

In [ ]:
from uuid import uuid4
products = {"1": ("USB-C Cable", 49900), "3": ("Water Bottle", 89900)}
def create_order(request):
    lines = []
    for item in request.items:
        name, price = products[item.product_id]
        lines.append({"product_id": item.product_id, "name": name,
                      "unit_price_paise": price, "quantity": item.quantity})
    return {"order_id": uuid4().hex, "lines": lines,
            "total_paise": sum(line["unit_price_paise"] * line["quantity"] for line in lines)}

sample_order = create_order(OrderRequest.model_validate(payload))
print(sample_order["total_paise"])


**Change and explain:** Which product choice passes structural validation but fails this lookup?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The real domain calculation returns 189700.

</details>

### 5. Save through the real transaction

A mock of storage could not establish atomic writes. Keep the full save real in this isolated database.

**Before running:** Predict the result or state change.

In [ ]:
def save_order(connection, order):
    with connection:
        connection.execute("INSERT INTO orders VALUES (?, ?)", (order["order_id"], order["total_paise"]))
        for position, line in enumerate(order["lines"], start=1):
            connection.execute("INSERT INTO order_lines VALUES (?, ?, ?, ?, ?, ?)",
                (order["order_id"], position, line["product_id"], line["name"],
                 line["unit_price_paise"], line["quantity"]))

with connect() as connection:
    save_order(connection, sample_order)
    print(connection.execute("SELECT COUNT(*) FROM order_lines").fetchone()[0])


**Change and explain:** Explain which context reverses earlier writes when a later insertion fails.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Two real snapshot lines are retained.

</details>

### 6. Wire the route to the connection seam

The supplied provider opens a fresh connection during the request. Real route decoding delegates to real domain and storage operations.

**Before running:** Predict the result or state change.

In [ ]:
from fastapi import FastAPI, HTTPException, Depends
from fastapi.testclient import TestClient
app = FastAPI()
def get_connection():
    with connect() as connection:
        yield connection

@app.post("/orders", status_code=201)
def place_order(request: OrderRequest, connection=Depends(get_connection)):
    try:
        order = create_order(request)
    except KeyError:
        raise HTTPException(404, "Unknown product")
    save_order(connection, order)
    return order

print(app.routes[-1].path)


**Change and explain:** Trace which parts will run for real when the client posts.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The registered public route is /orders; the next cells exercise it.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Read from storage rather than return cached memory

Retrieval translates retained rows. The handler cannot satisfy persistence by remembering the previous response alone.

**Before running:** Predict the result or state change.

In [ ]:
@app.get("/orders/{order_id}")
def get_order(order_id: str, connection=Depends(get_connection)):
    header = connection.execute("SELECT * FROM orders WHERE order_id = ?", (order_id,)).fetchone()
    if header is None:
        raise HTTPException(404, "Unknown order")
    rows = connection.execute(
        "SELECT product_id, name, unit_price_paise, quantity FROM order_lines WHERE order_id = ? ORDER BY line_number",
        (order_id,),
    )
    return {"order_id": order_id, "total_paise": header["total_paise"], "lines": [dict(row) for row in rows]}

with TestClient(app) as client:
    print(client.get("/orders/missing").status_code)


**Change and explain:** Which layer translates the absent row into an HTTP outcome?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

A missing order returns 404 through the real route.

</details>

### 8. Exercise the round trip and inspect stored rows

Response agreement and database observations support different parts of the integration claim.

**Before running:** Predict the result or state change.

In [ ]:
with TestClient(app) as client:
    created = client.post("/orders", json=payload)
    assert created.status_code == 201
    order_data = created.json()
    retrieved = client.get(f"/orders/{order_data['order_id']}")
    assert retrieved.json() == order_data
    assert order_data["total_paise"] == 189700
with connect() as connection:
    count = connection.execute("SELECT COUNT(*) FROM order_lines WHERE order_id = ?", (order_data["order_id"],)).fetchone()[0]
assert count == 2
print(created.status_code, order_data["total_paise"], count)


**Change and explain:** Each request opens storage again. What additional setup would test a freshly rebuilt application?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

201, total 189700, and two stored lines for this order.

</details>

### 9. Inject a later-line failure and inspect rollback

A server error alone cannot establish clean storage. Compare the resulting records with the earlier successful state.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    connection.executescript("""
        CREATE TRIGGER reject_bottle BEFORE INSERT ON order_lines
        WHEN NEW.product_id = '3'
        BEGIN SELECT RAISE(ABORT, 'simulated later-line failure'); END;
    """)
    before = tuple(connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
                   for table in ("orders", "order_lines"))
with TestClient(app, raise_server_exceptions=False) as client:
    failed = client.post("/orders", json=payload)
with connect() as connection:
    after = tuple(connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
                  for table in ("orders", "order_lines"))
assert failed.status_code == 500 and after == before
print(failed.status_code, before, after)
directory.cleanup()


**Change and explain:** The table names are supplied constants. Why does unchanged state establish more than checking 500?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The failed save adds neither a header nor lines; earlier successful orders remain. The temporary file is then removed.

</details>

### 10. Supply a real client contract at the external boundary

Client decisions can be explored separately from the local API. The response model still runs even when HTTP is substituted.

**Before running:** Predict the result or state change.

In [ ]:
import requests
from unittest.mock import Mock, patch

class OrderLineResponse(BaseModel):
    product_id: str
    name: str
    unit_price_paise: int = Field(strict=True, ge=0)
    quantity: int = Field(strict=True, gt=0)
class OrderResponse(BaseModel):
    order_id: str
    lines: list[OrderLineResponse] = Field(min_length=1)
    total_paise: int = Field(strict=True, ge=0)
def submit_order(base_url, payload):
    response = requests.post(f"{base_url}/orders", json=payload, timeout=5)
    response.raise_for_status()
    return OrderResponse.model_validate(response.json())

print(OrderResponse.model_validate(order_data).total_paise)


**Change and explain:** Locate the one external callable to replace in this notebook’s lookup context.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The detailed client response contract accepts the earlier real response.

</details>

### 11. Configure the response and verify the outgoing call

Return values control successful dependency behavior. Keep client parsing and validation real.

**Before running:** Predict the result or state change.

In [ ]:
response = Mock(spec=requests.Response)
response.raise_for_status.return_value = None
response.json.return_value = order_data
with patch("requests.post", return_value=response) as post:
    order = submit_order("http://shop.test", payload)
assert order.total_paise == 189700
post.assert_called_once_with("http://shop.test/orders", json=payload, timeout=5)
print(order.total_paise, post.call_count)


**Change and explain:** No request reached shop.test. Which claims still require real integration?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The client parses the controlled response and makes the expected outgoing call.

</details>

### 12. Observe controlled external failures

A timeout and a malformed response belong to different boundaries. Neither should become a successful created order.

**Before running:** Predict the result or state change.

In [ ]:
with patch("requests.post", side_effect=requests.Timeout("slow")):
    try:
        submit_order("http://shop.test", payload)
    except requests.Timeout:
        print("communication_failed")
response.json.return_value = {"order_id": "example", "total_paise": 99800}
with patch("requests.post", return_value=response):
    try:
        submit_order("http://shop.test", payload)
    except ValidationError:
        print("invalid_response")


**Change and explain:** Configure raise_for_status.side_effect with an HTTPError to explore a rejection next.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The real client propagates distinct failures; the M5 workflow records them with distinct statuses.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M8 lab guide](../../projects/nano-zon/labs/m08/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

### Apply the checks to project files

The [M8 source guide](../../teaching-snippets/m08/README.md) supplies the formal test/runner commands. The notebook is a small exploration; the Nano-Zon lab checks the complete application.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.